# 153. Find Minimum in Rotated Sorted Array
**Difficulty:** 🟡 Medium · **Topic:** Array · **LeetCode:** https://leetcode.com/problems/find-minimum-in-rotated-sorted-array/

## 💡 Concepts

**Core concept(s):** **Binary search** exploiting the rotation invariant.

**Why it applies here:** A sorted array rotated at some pivot still has a hidden order: one half around any midpoint is always properly sorted. Comparing `mid` to the **rightmost** element tells us which side the minimum (the rotation point) lies on, so we can discard half each step.

**Key intuition / mental model:** If `nums[mid] > nums[right]`, the dip (minimum) is to the **right** of mid; otherwise it's at mid or to its **left**. Narrow until the window is a single element.

---

### 📚 What is Binary Search?
**Binary search** finds a target (or a boundary) in a **sorted / monotonic** search space by repeatedly halving it: check the middle, then keep only the half that can contain the answer.
- **Complexity:** **O(log n)** time, **O(1)** space (iterative).
- **Key requirement:** a monotonic predicate — something that lets you decide which half to keep.

## 📝 Problem

A sorted array of **distinct** values was rotated at an unknown pivot. Return the minimum element in **O(log n)**.

**Example**
```
Input:  nums = [3, 4, 5, 1, 2]   Output: 1
Input:  nums = [4, 5, 6, 7, 0, 1, 2]   Output: 0
```
**Constraints:** all elements unique; `1 <= len(nums) <= 5000`.

> Two meaningfully distinct approaches: O(n) linear scan and the O(log n) binary search.

### Approach 1 — Linear Scan (worst)

**Idea:** Just take the minimum by scanning every element.

**Time complexity:** `O(n)`.

**Space complexity:** `O(1)`.

In [ ]:
from typing import List

def find_min_linear(nums: List[int]) -> int:
    m = nums[0]
    for x in nums:                         # just scan every value...
        m = min(m, x)                      # ...and keep the smallest
    return m

### Approach 2 — Binary Search (optimal)

**Idea:** Compare `nums[mid]` with `nums[right]`. If `nums[mid] > nums[right]`, the minimum is strictly right of `mid`; else it's `mid` or left. Converge to one element.

**Time complexity:** `O(log n)`.

**Space complexity:** `O(1)`.

In [ ]:
from typing import List

def find_min_binary(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1              # search range
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:           # middle bigger than right end...
            lo = mid + 1                   # ...the smallest must be to the RIGHT of mid
        else:
            hi = mid                       # smallest is at mid or to its LEFT
    return nums[lo]                        # lo == hi lands on the minimum

In [ ]:
# Correctness check
def rotate(sorted_list, k):
    k %= len(sorted_list)
    return sorted_list[k:] + sorted_list[:k]

tests = [
    ([3, 4, 5, 1, 2], 1),
    ([4, 5, 6, 7, 0, 1, 2], 0),
    ([11, 13, 15, 17], 11),                # no rotation
    ([2, 1], 1),
]
for nums, expected in tests:
    l, b = find_min_linear(nums), find_min_binary(nums)
    print(f"{nums} -> linear={l}, binary={b} | expected={expected}")
    assert l == b == expected, "mismatch!"

# extra: every rotation of 0..99 must yield min 0
base = list(range(100))
for k in range(100):
    assert find_min_binary(rotate(base, k)) == 0
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit) so the measurement reflects the true bound. Sub-millisecond rows are noisy — look at the trend, not one number.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    base = list(range(n))
    nums = base[n // 2:] + base[:n // 2]   # rotated sorted array of size n
    return (nums,)

solutions = {
    "linear O(n)    ": find_min_linear,
    "binary O(log n)": find_min_binary,
}
sizes = [2000, 4000, 8000, 16000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Binary search on a rotated/monotonic-ish array:** You don't need full sortedness — only a rule that reliably says which half to keep. Comparing `mid` to an endpoint provides it.
- **Search for a boundary, not a value:** Here we hunt the rotation point, converging `lo`/`hi` with `hi = mid` (not `mid-1`) to avoid skipping the answer.
- **Signal to reach for it:** "O(log n)", "sorted then rotated", "find pivot / minimum / first-true index".
- **Related problems:** Search in Rotated Sorted Array, Find Peak Element, First Bad Version, Koko Eating Bananas.
- **Common pitfalls:** (1) comparing to `nums[lo]` instead of `nums[hi]` (breaks on non-rotated input); (2) `hi = mid - 1` skipping the min; (3) infinite loop from wrong midpoint bias.